In [1]:
from pyaraucaria.obs_plan.obs_plan_parser import ObsPlanParser
from jsonschema import Draft7Validator
from copy import deepcopy
import re


In [2]:
def convert_to_obdict(ob):
    result = {"name": None, "ra": None, "dec": None, "type": None}

    subcommands = ob.get("subcommands", [])
    if isinstance(subcommands, dict):
        subcommands = [subcommands]

    for sub in subcommands:
        if "command_name" in sub:
            result["type"] = sub["command_name"]
        if "kwargs" in sub and isinstance(sub["kwargs"], dict):
            result.update(sub["kwargs"])
        if "args" in sub and isinstance(sub["args"], list):
            if len(sub["args"]) == 1:
                result["name"] = sub["args"][0]
            elif len(sub["args"]) == 3:
                result["name"] = sub["args"][0]
                result["ra"] = sub["args"][1]
                result["dec"] = sub["args"][2]

    return result



In [3]:
# optymalizacja

BASE_SCHEMA = {
    "type": "object",
    "properties": {

        "type": {"type": "string"},

        "name": {"type": ["string", "null"]},
        "seq": {"type": "string"},

        "sec": {"type": "number", "minimum": 0},
        "ut": {"type": "string", "pattern": r"^\d{1,2}:\d{2}:\d{2}$"},
        "sunrise": {"type": "number"},
        "sunset": {"type": "number"},

        "ra": {"type": ["string", "null"]},
        "dec": {"type": ["string", "null"]},

        "az": {"type": "number"},
        "alt": {"type": "number"},

        "epoch": {"type": "integer"},

        "gain": {"type": "string", "enum": ["low", "high"]},

        "mirror_cover": {
            "type": "string",
            "enum": ["auto", "open", "close"]
        },

        "auto_focus": {
            "type": "string",
            "enum": ["on", "off"]
        },

        "tracking": {
            "type": "string",
            "enum": ["on", "off"]
        },

        "dome_follow": {
            "type": "string",
            "enum": ["off"]
        },

        "read_mod": {
            "type": ["integer", "string"]
        },

        "focus_offset": {"type": "number"},

        "test": {"type": "boolean"},

        "observer": {"type": "string"},
        "uobi": {"type": "integer"},
        "sciprog": {"type": "string"},
        "pi": {"type": "string"},
    },

    "additionalProperties": True,
}


COMMAND_RULES = {

    "WAIT": {
        "required": {"type"},
        "allowed": {"type", "sec", "ut", "sunrise", "sunset","observer", "uobi" },
        "one_of_group": [["sec", "ut", "sunrise", "sunset"]]
    },

    "ZERO": {
        "required": {"type", "seq"},
        "allowed": {
            "type", "seq",
            "observer", "uobi",
            "gain", "test"
        }
    },

    "OBJECT": {
        "required": {"type"},
        "allowed": {
            "type", "name", "seq",
            "ra", "dec",
            "alt", "az",
            "observer", "uobi",
            "gain", "epoch"
        },
        "one_of": [
            {"ra", "dec"},
            {"alt", "az"}
        ],
        "additional": False
    }
}


In [4]:
def clean_none(obs):
    return {k: v for k, v in obs.items() if v is not None}

def convert_types(obs, schema):

    properties = schema.get("properties", {})
    converted = {}

    for key, value in obs.items():

        if key not in properties:
            converted[key] = value
            continue

        prop = properties[key]
        expected = prop.get("type")

        if not expected:
            converted[key] = value
            continue

        if not isinstance(expected, list):
            expected = [expected]

        new_value = value
        converted_successfully = False

        for typ in expected:

            try:
                if typ == "integer":
                    new_value = int(value)
                    converted_successfully = True
                    break

                elif typ == "number":
                    new_value = float(value)
                    converted_successfully = True
                    break

                elif typ == "boolean":
                    if isinstance(value, str):
                        if value.lower() in ["true", "1"]:
                            new_value = True
                            converted_successfully = True
                            break
                        if value.lower() in ["false", "0"]:
                            new_value = False
                            converted_successfully = True
                            break
                    elif isinstance(value, bool):
                        new_value = value
                        converted_successfully = True
                        break

                elif typ == "string":
                    new_value = str(value)
                    converted_successfully = True
                    break

                elif typ == "null":
                    if value is None:
                        new_value = None
                        converted_successfully = True
                        break

            except (ValueError, TypeError):
                pass

        # jeśli żadna konwersja nie przeszła → zostaw oryginał
        converted[key] = new_value if converted_successfully else value

    return converted




def build_schema(overrides=None):

    schema = deepcopy(BASE_SCHEMA)  # robimy kopię, żeby nie zmieniać globalnej BASE_SCHEMA

    if overrides:
        for key, override in overrides.items():
            if key in schema["properties"]:
                schema["properties"][key].update(override)

    return schema

def validate_types(obs, schema):

    validator = Draft7Validator(schema)

    # wynik tylko dla pól, które podał użytkownik
    result = {key: True for key in obs}

    for error in validator.iter_errors(obs):
        # ignorujemy required i oneOf
        if error.validator in ("required", "oneOf"):
            continue

        # przypisujemy błąd do konkretnego pola tylko jeśli istnieje w obs
        if error.path:
            key = error.path[0]
            if key in result:
                result[key] = False

    return result


def validate_rules(obs, rules):
    """
    Walidacja required, one_of i one_of_group w jednym kroku.

    obs   = słownik obserwacji (tylko podane pola)
    rules = COMMAND_RULES[cmd_type]

    Zwraca słownik błędów dla pól:
        - None  -> brak pola wymagane
        - False -> oneOf / one_of_group nie spełnione
    """
    result = {}

    # --- 1. required ---
    for key in rules.get("required", []):
        if key not in obs:
            result[key] = None

    # --- 2. one_of (klasyczne podschematy JSON Schema) ---
    one_of = rules.get("one_of", [])
    if one_of:
        satisfied = False
        for subschema in one_of:
            if all(k in obs for k in subschema):
                satisfied = True
                break
        if not satisfied:
            # żaden podschemat nie spełniony → wszystkie pola z każdego podschematu False
            for subschema in one_of:
                for k in subschema:
                    result[k] = False

    # --- 3. one_of_group (grupy logiczne np. WAIT) ---
    for group in rules.get("one_of_group", []):
        if not any(k in obs for k in group):
            for k in group:
                result[k] = False

    return result





In [5]:
def validate_radec(obs):
    """
    Walidacja pól RA, DEC .
    Zwraca słownik błędów:
        True  -> poprawne
        False -> niepoprawne
    """
    result = {}

    # --- RA ---
    ra = obs.get("ra")
    if ra is not None:
        # hh:mm:ss.ss
        if re.match(r"^\d{1,2}:\d{2}:\d{2}(\.\d+)?$", ra):
            result["ra"] = True
        else:
            result["ra"] = False

    # --- DEC ---
    dec = obs.get("dec")
    if dec is not None:
        # ±dd:mm:ss.ss
        if re.match(r"^[\+\-]?\d{1,2}:\d{2}:\d{2}(\.\d+)?$", dec):
            result["dec"] = True
        else:
            result["dec"] = False

    return result



In [6]:

def validate_seq(obs, allowed_filters=None):
    """
    Walidacja pola 'seq' w słowniku obs.
    Obsługuje:
      - pojedyncze ekspozycje: 5/I/20
      - listę ekspozycji: 5/Ic/60,5/V/70
      - powtarzanie sekwencji: 10x(5/Ic/60,5/V/70)
    """
    result = {}
    allowed_filters = allowed_filters or []

    seq_str = obs.get("seq")
    if seq_str is None:
        return result  # brak pola seq → nic nie robimy

    def _validate(seq_str):
        seq_str = seq_str.strip()

        # --- powtarzanie Nx(...) ---
        match = re.match(r"^(\d+)x\((.+)\)$", seq_str)
        if match:
            repeat_count = int(match.group(1))
            inner = match.group(2)
            if repeat_count < 1:
                return False
            return _validate(inner)

        # --- lista ekspozycji rozdzielona przecinkiem ---
        exposures = [s.strip() for s in seq_str.split(",")]
        for exp in exposures:
            parts = exp.split("/")
            if len(parts) != 3:
                return False
            try:
                n_repeat = int(parts[0])
                filt = parts[1]
                exp_time = float(parts[2].replace(",", "."))
                if n_repeat < 1 or exp_time < 0:
                    return False
                if allowed_filters and filt not in allowed_filters:
                    return False
            except ValueError:
                return False
        return True

    # --- wykonanie walidacji ---
    result["seq"] = _validate(seq_str)
    return result

In [7]:
def validate_ob(ob,schema,rules, overrides=None):
    # --- 1. clean ---
    obs = clean_none(ob)

    # --- 2. schema ---
    schema = build_schema(overrides)

    # --- 3. convert ---
    obs = convert_types(obs, schema)

    # --- 4. walidacja typów ---
    result = validate_types(obs, schema)

    # --- 5. walidacja required / oneOf ---
    cmd_type = obs.get("type")
    if cmd_type in COMMAND_RULES:
        result.update(validate_rules(obs, rules[cmd_type]))
    else:
        result["type"] = False  # nieznana komenda

    # --- 6. ra dec ---
    result.update(validate_radec(obs))

    # --- 7. sekwencja ---
    result.update(validate_seq(obs,allowed_filters=["Ic","V","B"]))
    
    # --- 8. valid tylko jeśli brak False lub None ---
    valid = all(v is True for v in result.values())

    return {
        "valid": valid,
        "result": result,
        "data": obs
    }

In [24]:
# txt = "OBJECT V496_Aql 19:08:20.77 -07:26:15.89 seq=1/V/20 \n WAIT ut=16:00:00"
# txt = "WAIT ut=16:00:00"
# txt = "WAIT sunrise=-10"
# txt = "WAIT sec=300"
# txt = "ZERO seq=15/Ic/0"
# txt = "DARK ZZ01 seq=10/V/300,10/Ic/200"
# txt = "DOMEFLAT AL3 seq=10/i/100"
# txt = "SKYFLAT HD23 alt=60.0 az=270.0 seq=10/r/20,10/V/30"
# txt = "SKYFLAT HD24 seq=10/g/a,10/V/a"
# txt = "FOCUS NG31 12:12:12 20:20:20"
# txt = "OBJECT HD193901 20:23:35.8 -21:22:14.0 seq=1/V/300"
# txt = "OBJECT FF_Aql 18:58:14.75 17:21:39.29 seq=5/Ic/60,5/V/70"
# txt = "OBJECT V496_Aql 19:08:20.77 -07:26:15.89 seq=1/V/20"
# txt = "OBJECT V496_Aql seq=1/V/20"
# txt = "OBJECT HD23 alt=dupa az=270.0"
# txt = "OBJECT V496_Aql 19:08:20.77 "
# txt = "OBJECT V496_Aql 19:08:20.77 -07:26:15.89 alt=34 az=270.0 seq=1/V/20"
# txt = "WAIT"
# txt = "OBJECT HD193901 20:23:35.8 -21:22:14.0 seq=1/dupa/300"
# txt = "OBJECT HD193901 20:23:35.8 -21:22:14.0 seq=10x(1/Ic/300,/V/10)"
txt = "OBJECT HD193901 20:23:35.8 -21:30 seq=10x(1/Ic/300,3/V/10)"


ob_tmp = ObsPlanParser.convert_from_string(txt)
ob = convert_to_obdict(ob_tmp)
validate_ob(ob,BASE_SCHEMA,COMMAND_RULES)


{'valid': False,
 'result': {'name': True, 'ra': True, 'dec': False, 'type': True, 'seq': True},
 'data': {'name': 'HD193901',
  'ra': '20:23:35.8',
  'dec': '-21:30',
  'type': 'OBJECT',
  'seq': '10x(1/Ic/300,3/V/10)'}}

In [ ]:
#